In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


In [25]:

df = pd.read_csv("dataset/SQLiV3.csv")

print(df.shape)
print(df.columns)
print(df.head())
print(df.info())

df["Label"] = pd.to_numeric(df["Label"], errors="coerce")
print(df["Label"].isnull().sum())

df = df[df["Label"].isin([0, 1])]

df = df[["Sentence", "Label"]]

print(df.info())


(30919, 4)
Index(['Sentence', 'Label', 'Unnamed: 2', 'Unnamed: 3'], dtype='str')
                                            Sentence Label Unnamed: 2  \
0                  " or pg_sleep  (  __TIME__  )  --     1        NaN   
1  create user name identified by pass123 tempora...   NaN          1   
2   AND 1  =  utl_inaddr.get_host_address   (    ...     1        NaN   
3   select * from users where id  =  '1' or @ @1 ...     1        NaN   
4   select * from users where id  =  1 or 1#"  ( ...     1        NaN   

   Unnamed: 3  
0         NaN  
1         NaN  
2         NaN  
3         NaN  
4         NaN  
<class 'pandas.DataFrame'>
RangeIndex: 30919 entries, 0 to 30918
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Sentence    30904 non-null  str    
 1   Label       30664 non-null  str    
 2   Unnamed: 2  306 non-null    str    
 3   Unnamed: 3  9 non-null      float64
dtypes: float64(1), str(3)
memory usage: 

In [26]:
df["Sentence"] = df["Sentence"].astype(str)
df = df.drop_duplicates(subset=["Sentence"])
print(df.shape)

(30595, 2)


In [27]:
print(df["Label"].value_counts())
print(df["Label"].value_counts(normalize=True))

Label
0.0    19258
1.0    11337
Name: count, dtype: int64
Label
0.0    0.629449
1.0    0.370551
Name: proportion, dtype: float64


In [28]:
X = df["Sentence"]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True))

Training samples: 24476
Testing samples: 6119

Training distribution:
Label
0.0    0.629433
1.0    0.370567
Name: proportion, dtype: float64

Testing distribution:
Label
0.0    0.629515
1.0    0.370485
Name: proportion, dtype: float64


In [32]:
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=100000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)

Training shape: (24476, 99544)
Testing shape: (6119, 99544)


In [33]:
print(X_train_tfidf.shape)
print(vectorizer.get_feature_names_out()[:50])


(24476, 99544)
['\x18 ' '\x18 o' '\x18 or' '\x18 or ' ' !' ' !<' ' !<1' ' !<1,' ' !<@'
 ' !<@ ' ' !<@,' ' "' ' " ' ' " "' ' " ",' ' " (' ' " ( ' ' " )' ' " ) '
 ' " =' ' " = ' ' " o' ' " or' ' ""' ' "" ' ' "" =' ' "$' ' "%' ' "%"'
 ' "%" ' ' "%m' ' "%m ' ' "%w' ' "%w ' ' "&' ' "& ' ' ",' ' ", ' ' "-'
 ' ".' ' "."' ' ".",' ' "0' ' "1' ' "1"' ' "1" ' ' "1"#' ' "1"-' ' "1"/'
 ' "10']


In [35]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [36]:
y_pred = model.predict(X_test_tfidf)

In [37]:
print(y_pred[:20])
print(y_test.values[:20])

[1. 1. 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1.]
[1. 1. 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1.]


In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal SQL", "SQL Injection"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.9947703873181892
Precision: 0.9991067440821796
Recall   : 0.9867666519629467
F1 Score : 0.9928983577452286

Classification Report:
               precision    recall  f1-score   support

   Normal SQL       0.99      1.00      1.00      3852
SQL Injection       1.00      0.99      0.99      2267

     accuracy                           0.99      6119
    macro avg       1.00      0.99      0.99      6119
 weighted avg       0.99      0.99      0.99      6119


Confusion Matrix:
[[3850    2]
 [  30 2237]]


In [39]:
errors = df.loc[X_test.index][y_test != y_pred]

print(errors[["Sentence", "Label"]].head(50).to_string(index=False))

                                                                                                                                                                                                                                                                                                Sentence  Label
                                                                                                                                                                                                                               SELECT * FROM Users WHERE Name  = "" or "" = "" AND Pass  = "" or "" = ""    0.0
                                                                                                                                                                                                                                                                                                  insert    1.0
                                                                                        

In [40]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

top_attack = np.argsort(coefficients)[-30:][::-1]
top_normal = np.argsort(coefficients)[:30]

print("Strongest SQL injection features:")
for i in top_attack:
    print(repr(feature_names[i]), coefficients[i])

print("\nStrongest normal SQL features:")
for i in top_normal:
    print(repr(feature_names[i]), coefficients[i])

Strongest SQL injection features:
' )' 5.482145422048287
') ' 5.482145422048287
' ) ' 5.482145422048287
' = ' 5.291940952137884
'= ' 5.291940952137884
' =' 5.291940952137884
'--' 4.430967941394003
'( ' 3.7590355246885214
' ( ' 3.7590355246885214
' (' 3.7590355246885214
'1 ' 3.7501286537898637
'" ' 3.5705441820546095
' 1' 3.5110278406122784
"' " 3.420774961913007
'or ' 3.409235399061283
"1'" 3.3206311276745373
',4' 3.110583782993513
' or ' 2.8742504604682817
',3' 2.8508881191407527
',8' 2.7926332590545537
' 9' 2.7822565665988392
',1' 2.757054262633414
'r ' 2.7362512142556983
' 8' 2.7142281119513267
',6' 2.6569974851504456
',9' 2.6093679578338587
',2' 2.602980747109093
' all ' 2.5940303576045456
' 5' 2.5817085708700067
' 6' 2.571281529570246

Strongest normal SQL features:
', ' -2.9716316568248224
'ro' -2.8774936980931027
' f' -2.6564157164011983
'm ' -2.3399903223248364
'fr' -2.3160914002110777
'ar' -2.2779404865094293
'om' -2.27116628691377
'rom' -2.212658590385709
'om ' -2.21075346685

In [41]:
test_queries = [
    # Normal SQL
    "SELECT * FROM users WHERE id = 10",
    "SELECT name, email FROM customers WHERE age > 18",
    "SELECT id FROM products WHERE price < 500",
    "UPDATE users SET name = 'John' WHERE id = 5",
    "DELETE FROM users WHERE id = 100",
    "SELECT COUNT(*) FROM orders",
    "SELECT * FROM employees ORDER BY salary DESC",

    # SQL Injection
    "' OR 1=1 --",
    "' OR '1'='1' --",
    "' UNION SELECT username, password FROM users --",
    "'; DROP TABLE users; --",
    "' OR 1=1 #",
    "admin' --",
    "' AND 1=2 UNION SELECT NULL --",
    "' OR SLEEP(5) --"
]

test_vectors = vectorizer.transform(test_queries)
predictions = model.predict(test_vectors)
probabilities = model.predict_proba(test_vectors)

for query, prediction, probability in zip(
    test_queries,
    predictions,
    probabilities
):
    label = "SQL Injection" if prediction == 1 else "Normal SQL"
    confidence = max(probability) * 100

    print(f"{label:15} | {confidence:6.2f}% | {query}")

SQL Injection   |  59.36% | SELECT * FROM users WHERE id = 10
Normal SQL      |  95.99% | SELECT name, email FROM customers WHERE age > 18
Normal SQL      |  97.70% | SELECT id FROM products WHERE price < 500
SQL Injection   |  61.39% | UPDATE users SET name = 'John' WHERE id = 5
SQL Injection   |  55.19% | DELETE FROM users WHERE id = 100
Normal SQL      |  99.73% | SELECT COUNT(*) FROM orders
Normal SQL      |  99.62% | SELECT * FROM employees ORDER BY salary DESC
SQL Injection   |  98.13% | ' OR 1=1 --
SQL Injection   |  94.22% | ' OR '1'='1' --
SQL Injection   |  62.06% | ' UNION SELECT username, password FROM users --
Normal SQL      |  66.83% | '; DROP TABLE users; --
SQL Injection   |  96.72% | ' OR 1=1 #
SQL Injection   |  57.48% | admin' --
SQL Injection   |  98.43% | ' AND 1=2 UNION SELECT NULL --
SQL Injection   |  94.11% | ' OR SLEEP(5) --


In [42]:
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=100000
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_test_char = char_vectorizer.transform(X_test)

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    min_df=2,
    max_features=50000,
    sublinear_tf=True
)

X_train_word = word_vectorizer.fit_transform(X_train)
X_test_word = word_vectorizer.transform(X_test)

print("Character:", X_train_char.shape)
print("Word:", X_train_word.shape)

Character: (24476, 99544)
Word: (24476, 20939)


In [44]:
from scipy.sparse import hstack

X_train_combined = hstack([
    X_train_char,
    X_train_word
]).tocsr()

X_test_combined = hstack([
    X_test_char,
    X_test_word
]).tocsr()

print("Combined:", X_train_combined.shape)

Combined: (24476, 120483)


In [45]:
from sklearn.linear_model import LogisticRegression

combined_model = LogisticRegression(
    max_iter=1000,
    C=2.0,
    random_state=42
)

combined_model.fit(X_train_combined, y_train)

y_pred_combined = combined_model.predict(X_test_combined)

In [46]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    y_test,
    y_pred_combined,
    target_names=["Normal SQL", "SQL Injection"]
))

print(confusion_matrix(y_test, y_pred_combined))

               precision    recall  f1-score   support

   Normal SQL       0.99      1.00      1.00      3852
SQL Injection       1.00      0.99      1.00      2267

     accuracy                           1.00      6119
    macro avg       1.00      1.00      1.00      6119
 weighted avg       1.00      1.00      1.00      6119

[[3850    2]
 [  20 2247]]


In [47]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(
    n_components=300,
    random_state=42
)

X_train_svd = svd.fit_transform(X_train_combined)
X_test_svd = svd.transform(X_test_combined)

print(X_train_svd.shape)
print(X_test_svd.shape)

(24476, 300)
(6119, 300)


In [49]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_svd, y_train)

y_pred_xgb = xgb_model.predict(X_test_svd)

In [50]:
print(classification_report(
    y_test,
    y_pred_xgb,
    target_names=["Normal SQL", "SQL Injection"]
))

print(confusion_matrix(y_test, y_pred_xgb))

               precision    recall  f1-score   support

   Normal SQL       1.00      1.00      1.00      3852
SQL Injection       1.00      1.00      1.00      2267

     accuracy                           1.00      6119
    macro avg       1.00      1.00      1.00      6119
 weighted avg       1.00      1.00      1.00      6119

[[3846    6]
 [  11 2256]]


In [51]:
p_char = model.predict_proba(X_test_tfidf)[:, 1]

p_combined = combined_model.predict_proba(
    X_test_combined
)[:, 1]

p_xgb = xgb_model.predict_proba(
    X_test_svd
)[:, 1]

In [52]:
import numpy as np

ensemble_probability = (
    0.35 * p_char +
    0.35 * p_combined +
    0.30 * p_xgb
)

y_pred_ensemble = (
    ensemble_probability >= 0.5
).astype(int)

In [53]:
print(classification_report(
    y_test,
    y_pred_ensemble,
    target_names=["Normal SQL", "SQL Injection"]
))

print(confusion_matrix(y_test, y_pred_ensemble))

               precision    recall  f1-score   support

   Normal SQL       1.00      1.00      1.00      3852
SQL Injection       1.00      0.99      1.00      2267

     accuracy                           1.00      6119
    macro avg       1.00      1.00      1.00      6119
 weighted avg       1.00      1.00      1.00      6119

[[3850    2]
 [  17 2250]]


In [54]:
def detect_sql_injection(query):
    char_features = char_vectorizer.transform([query])
    word_features = word_vectorizer.transform([query])

    combined = hstack([
        char_features,
        word_features
    ]).tocsr()

    reduced = svd.transform(combined)

    p1 = model.predict_proba(char_features)[0, 1]
    p2 = combined_model.predict_proba(combined)[0, 1]
    p3 = xgb_model.predict_proba(reduced)[0, 1]

    probability = (
        0.35 * p1 +
        0.35 * p2 +
        0.30 * p3
    )

    if probability >= 0.5:
        return {
            "prediction": "SQL Injection",
            "probability": probability
        }

    return {
        "prediction": "Safe",
        "probability": 1 - probability
    }

In [55]:
print(detect_sql_injection("' OR 1=1 --"))

{'prediction': 'SQL Injection', 'probability': np.float64(0.9934234530041959)}


In [56]:
print(detect_sql_injection(
    "SELECT name FROM users WHERE id = 10"
))

{'prediction': 'SQL Injection', 'probability': np.float64(0.8024225511137975)}


In [57]:
query = "SELECT name FROM users WHERE id = 10"

char_features = char_vectorizer.transform([query])
word_features = word_vectorizer.transform([query])

combined = hstack([
    char_features,
    word_features
]).tocsr()

reduced = svd.transform(combined)

p_char = model.predict_proba(char_features)[0, 1]
p_combined = combined_model.predict_proba(combined)[0, 1]
p_xgb = xgb_model.predict_proba(reduced)[0, 1]

ensemble = (
    0.35 * p_char +
    0.35 * p_combined +
    0.30 * p_xgb
)

print("Character LR :", p_char)
print("Combined LR  :", p_combined)
print("XGBoost      :", p_xgb)
print("Ensemble     :", ensemble)

Character LR : 0.7024757984082773
Combined LR  : 0.7683462912139514
XGBoost      : 0.9587827
Ensemble     : 0.8024225511137975


In [72]:
import re
import numpy as np
import pandas as pd


def extract_sql_features(query):
    q = query.lower()

    features = {
        # Basic properties
        "length": len(query),
        "word_count": len(query.split()),

        # Special characters
        "single_quotes": query.count("'"),
        "double_quotes": query.count('"'),
        "backticks": query.count("`"),
        "semicolons": query.count(";"),
        "equals": query.count("="),
        "parentheses": query.count("(") + query.count(")"),
        "commas": query.count(","),
        "slashes": query.count("/"),

        # SQL operators
        "or_count": len(re.findall(r"\bor\b", q)),
        "and_count": len(re.findall(r"\band\b", q)),
        "between_count": len(re.findall(r"\bbetween\b", q)),

        # SQL keywords
        "select_count": len(re.findall(r"\bselect\b", q)),
        "from_count": len(re.findall(r"\bfrom\b", q)),
        "where_count": len(re.findall(r"\bwhere\b", q)),
        "union_count": len(re.findall(r"\bunion\b", q)),
        "insert_count": len(re.findall(r"\binsert\b", q)),
        "update_count": len(re.findall(r"\bupdate\b", q)),
        "delete_count": len(re.findall(r"\bdelete\b", q)),
        "drop_count": len(re.findall(r"\bdrop\b", q)),
        "truncate_count": len(re.findall(r"\btruncate\b", q)),
        "create_count": len(re.findall(r"\bcreate\b", q)),
        "alter_count": len(re.findall(r"\balter\b", q)),

        # Common injection indicators
        "comment_count": len(re.findall(r"--|/\*|\*/|#", query)),
        "hex_count": len(re.findall(r"0x[0-9a-f]+", q)),
        "sleep_count": len(re.findall(r"\bsleep\b", q)),
        "benchmark_count": len(re.findall(r"\bbenchmark\b", q)),
        "xp_cmdshell_count": len(re.findall(r"xp_cmdshell", q)),

        "statement_count": len([
            x for x in query.split(";")
            if x.strip()
        ]),

        "semicolon_after_quote": int(
            bool(re.search(r"['\"].*;", query, re.DOTALL))
        ),

        "comment_after_semicolon": int(
            bool(re.search(r";.*(--|#|/\*)", query, re.DOTALL))
        ),

        "quote_before_comment": int(
            bool(re.search(r"['\"].*(--|#|/\*)", query, re.DOTALL))
        ),

        # Boolean patterns
        "tautology_count": len(
            re.findall(
                r"\b\d+\s*=\s*\d+\b",
                q
            )
        ),
    }

    return list(features.values())

In [73]:
normal = "SELECT name FROM users WHERE id = 10"

attack = "' OR 1=1 --"

print(extract_sql_features(normal))
print(extract_sql_features(attack))

[36, 8, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
[11, 4, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1]


In [74]:
X_train_sql = np.array([
    extract_sql_features(q)
    for q in X_train
])

X_test_sql = np.array([
    extract_sql_features(q)
    for q in X_test
])

print(X_train_sql.shape)
print(X_test_sql.shape)

(24476, 34)
(6119, 34)


In [75]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_sql_scaled = scaler.fit_transform(X_train_sql)
X_test_sql_scaled = scaler.transform(X_test_sql)

In [76]:
from scipy.sparse import csr_matrix, hstack

X_train_final = hstack([
    X_train_combined,
    csr_matrix(X_train_sql_scaled)
]).tocsr()

X_test_final = hstack([
    X_test_combined,
    csr_matrix(X_test_sql_scaled)
]).tocsr()

print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

Final training shape: (24476, 120517)
Final testing shape: (6119, 120517)


In [77]:
from sklearn.linear_model import LogisticRegression

final_model = LogisticRegression(
    max_iter=1500,
    C=2.0,
    random_state=42
)

final_model.fit(X_train_final, y_train)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",2.0
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1500
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [78]:
y_pred_final = final_model.predict(X_test_final)
y_prob_final = final_model.predict_proba(X_test_final)[:, 1]

In [79]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy :", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall   :", recall_score(y_test, y_pred_final))
print("F1 Score :", f1_score(y_test, y_pred_final))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_final,
    target_names=["Normal SQL", "SQL Injection"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

Accuracy : 0.9959143650923353
Precision: 0.9982222222222222
Recall   : 0.9907366563740626
F1 Score : 0.9944653531104716

Classification Report:
               precision    recall  f1-score   support

   Normal SQL       0.99      1.00      1.00      3852
SQL Injection       1.00      0.99      0.99      2267

     accuracy                           1.00      6119
    macro avg       1.00      0.99      1.00      6119
 weighted avg       1.00      1.00      1.00      6119


Confusion Matrix:
[[3848    4]
 [  21 2246]]


In [80]:
def predict_query(query):
    char_features = char_vectorizer.transform([query])
    word_features = word_vectorizer.transform([query])

    tfidf_features = hstack([
        char_features,
        word_features
    ]).tocsr()

    sql_features = np.array([
        extract_sql_features(query)
    ])

    sql_features = scaler.transform(sql_features)

    final_features = hstack([
        tfidf_features,
        csr_matrix(sql_features)
    ]).tocsr()

    probability = final_model.predict_proba(
        final_features
    )[0, 1]

    prediction = int(probability >= 0.5)

    return {
        "prediction": (
            "SQL Injection"
            if prediction == 1
            else "Safe"
        ),
        "probability": float(probability)
    }

In [81]:
print(
    predict_query(
        "SELECT name FROM users WHERE id = 10"
    )
)

{'prediction': 'Safe', 'probability': 0.015565153941965225}


In [82]:
queries = [
    "SELECT name FROM users WHERE id = 10",
    "SELECT name, email FROM customers WHERE age > 18",
    "UPDATE users SET name = 'John' WHERE id = 5",
    "DELETE FROM users WHERE id = 100",
    "' OR 1=1 --",
    "' UNION SELECT username, password FROM users --",
    "admin' --",
    "' OR SLEEP(5) --"
]

for q in queries:
    print(predict_query(q), "|", q)

{'prediction': 'Safe', 'probability': 0.015565153941965225} | SELECT name FROM users WHERE id = 10
{'prediction': 'Safe', 'probability': 0.0004150076787507607} | SELECT name, email FROM customers WHERE age > 18
{'prediction': 'Safe', 'probability': 0.07392708116659996} | UPDATE users SET name = 'John' WHERE id = 5
{'prediction': 'Safe', 'probability': 0.0014310204275203835} | DELETE FROM users WHERE id = 100
{'prediction': 'SQL Injection', 'probability': 0.9999999979127321} | ' OR 1=1 --
{'prediction': 'SQL Injection', 'probability': 0.9999587804875291} | ' UNION SELECT username, password FROM users --
{'prediction': 'SQL Injection', 'probability': 0.999894205090065} | admin' --
{'prediction': 'SQL Injection', 'probability': 0.9999999416508171} | ' OR SLEEP(5) --


In [83]:
hard_tests = [
    # Normal SQL
    "SELECT * FROM users WHERE username = 'admin'",
    "SELECT * FROM users WHERE id = 1 AND status = 'active'",
    "SELECT name FROM employees WHERE department = 'IT'",
    "INSERT INTO users (name, email) VALUES ('John', 'john@test.com')",
    "UPDATE users SET email = 'test@example.com' WHERE id = 42",

    # Basic injection
    "' OR 1=1 --",
    "' AND 1=2 --",
    "' OR 'a'='a' --",

    # UNION injection
    "' UNION SELECT username, password FROM users --",
    "1 UNION SELECT NULL,NULL,NULL --",

    # Comment variations
    "admin'--",
    "admin' #",
    "admin' /*",

    # Spacing/obfuscation
    "' OR 1 = 1 --",
    "'/**/OR/**/1=1--",
    "' OR/**/1=1 --",

    # Semicolon
    "'; DROP TABLE users; --",

    # Boolean manipulation
    "1 AND 1=1",
    "1 AND 1=2",

    # Time-based
    "' OR SLEEP(5) --",
    "'; WAITFOR DELAY '00:00:05' --"
]

for q in hard_tests:
    result = predict_query(q)
    print(
        f"{result['prediction']:15} "
        f"{result['probability']:.4f} | {q}"
    )

Safe            0.0114 | SELECT * FROM users WHERE username = 'admin'
Safe            0.0052 | SELECT * FROM users WHERE id = 1 AND status = 'active'
Safe            0.0017 | SELECT name FROM employees WHERE department = 'IT'
Safe            0.0005 | INSERT INTO users (name, email) VALUES ('John', 'john@test.com')
Safe            0.0082 | UPDATE users SET email = 'test@example.com' WHERE id = 42
SQL Injection   1.0000 | ' OR 1=1 --
SQL Injection   1.0000 | ' AND 1=2 --
SQL Injection   1.0000 | ' OR 'a'='a' --
SQL Injection   1.0000 | ' UNION SELECT username, password FROM users --
SQL Injection   0.9996 | 1 UNION SELECT NULL,NULL,NULL --
SQL Injection   0.9999 | admin'--
SQL Injection   0.9999 | admin' #
SQL Injection   0.9999 | admin' /*
SQL Injection   1.0000 | ' OR 1 = 1 --
SQL Injection   1.0000 | '/**/OR/**/1=1--
SQL Injection   1.0000 | ' OR/**/1=1 --
SQL Injection   1.0000 | '; DROP TABLE users; --
SQL Injection   0.9988 | 1 AND 1=1
SQL Injection   0.9988 | 1 AND 1=2
SQL Injecti